# Session 22: Cross-Validation and Overfitting

Data Science with Python, Level 1, Stage 3: Statistics and ML Fundamentals

## Learning Objectives

By the end of this session you will be able to:

1. Explain why one train and test split is not enough.
2. Use k-fold cross-validation with cross_val_score.
3. Recognise overfitting and underfitting.
4. Compare training and validation scores.
5. Use a pipeline so scaling happens inside each fold.
6. Reduce overfitting with simpler models or regularisation.

## Where We Are

In Session 21 every metric came from **one** random split of the data. Change `random_state` and the numbers change too. Today you learn **cross-validation**, which tests a model on several splits and averages the results.

You will also meet the most important danger in machine learning: **overfitting**, when a model memorises the training data and fails on new data. This is the last teaching session of Level 1. Next session, Project 3, puts every tool of Stage 3 together.

## Setup

Run this cell first. It loads the libraries and sets the address of the course datasets.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, r2_score

sns.set_theme(style="whitegrid")
DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"

churn = pd.read_csv(DATA_URL + "telecom_churn.csv")
churn["left"] = (churn["churn"] == "Yes").astype(int)
churn["monthly_contract"] = (churn["contract"] == "Month-to-month").astype(int)
churn["fiber"] = (churn["internet"] == "Fiber").astype(int)
churn["echeck"] = (churn["payment_method"] == "Electronic check").astype(int)
features = ["tenure_months", "monthly_charges", "support_calls", "senior",
            "monthly_contract", "fiber", "echeck"]
X = churn[features].astype(float)
y = churn["left"]
bikes = pd.read_csv(DATA_URL + "bike_rentals.csv")

## 1. One Split Is Luck

A **pipeline** joins steps into one model. `make_pipeline(StandardScaler(), LogisticRegression())` scales the data and then fits the classifier, and it remembers the scaling for new data. Level 2 covers pipelines in depth; today we use them because they make cross-validation safe.

In [ ]:
for seed in range(5):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=seed, stratify=y)
    model = make_pipeline(StandardScaler(), LogisticRegression())
    model.fit(X_train, y_train)
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    print("split", seed, "AUC", round(auc, 3))

## 2. K-Fold Cross-Validation

In [ ]:
model = make_pipeline(StandardScaler(), LogisticRegression())
scores = cross_val_score(model, X, y, cv=5, scoring="roc_auc")

print("AUC per fold:", scores.round(3))
print("Mean:", round(scores.mean(), 3), " Std:", round(scores.std(), 3))

| scoring= | Metric | Used for |
| --- | --- | --- |
| "accuracy" | Accuracy | Classification, the default |
| "roc_auc" | ROC AUC | Binary classification |
| "f1", "recall" | F1, recall | Rare positive class |
| "r2" | R2 | Regression, the default |
| "neg_mean_absolute_error" | Minus MAE | Regression; higher is better, so it is negative |

For classification, cross_val_score uses stratified folds automatically.

## 3. Overfitting

To see overfitting clearly, train on only **25 days** of bike rentals and predict rentals from temperature with curves of growing flexibility. `PolynomialFeatures(degree)` adds `temp squared`, `temp cubed` and so on, so the line can bend more and more.

In [ ]:
train = bikes.sample(25, random_state=2)
test = bikes.drop(train.index)

def poly_model(degree):
    return make_pipeline(PolynomialFeatures(degree), StandardScaler(),
                         LinearRegression())

In [ ]:
for d in [1, 2, 4, 8, 12]:
    m = poly_model(d).fit(train[["temp_c"]], train["rentals"])
    r2_train = m.score(train[["temp_c"]], train["rentals"])
    r2_test = m.score(test[["temp_c"]], test["rentals"])
    print(f"degree {d:2}: train {r2_train:.2f}  test {r2_test:.2f}")

In [ ]:
grid = pd.DataFrame({"temp_c": np.linspace(0, 30, 200)})
plt.scatter(train["temp_c"], train["rentals"], color="black")
for d in [1, 12]:
    m = poly_model(d).fit(train[["temp_c"]], train["rentals"])
    plt.plot(grid["temp_c"], m.predict(grid), label=f"degree {d}")
plt.ylim(0, 9000)
plt.legend()
plt.show()

## 4. Seeing the Gap with cross_validate

In [ ]:
sample = bikes.sample(60, random_state=0)
for d in [1, 3, 10]:
    res = cross_validate(poly_model(d), sample[["temp_c"]], sample["rentals"],
                         cv=5, return_train_score=True)
    print(d, "train", res["train_score"].mean().round(2),
          "validation", res["test_score"].mean().round(2))

In [ ]:
for alpha in [0, 1, 10]:
    reg = Ridge(alpha=alpha) if alpha else LinearRegression()
    model = make_pipeline(PolynomialFeatures(12), StandardScaler(), reg)
    model.fit(train[["temp_c"]], train["rentals"])
    r2_test = r2_score(test["rentals"], model.predict(test[["temp_c"]]))
    print("alpha", alpha, "test R2", round(r2_test, 2))

## Common Mistakes

| Symptom | Mistake | Fix |
| --- | --- | --- |
| Scaling leak | Scaling all the data before cross-validation | Put the scaler inside a pipeline so each fold fits its own. |
| Tuning on test | Trying many models and picking the best test score | Choose with cross-validation; touch the test set once at the end. |
| Only the mean | Reporting the mean score without its spread | Report mean and std across folds. |
| Ignoring the gap | Celebrating a 0.99 training score | Check the validation score; a big gap means overfitting. |

## In-Class Exercises

Work through these exercises during the session. Write your answer in the code cell under each one.

### In-Class Exercise 1: Ten Folds

Run `cross_val_score` on the churn pipeline with `cv=10` and `scoring="accuracy"`. Print the mean and std. Compare with the baseline of always predicting stays.

In [ ]:
# Write your code below

### In-Class Exercise 2: Fewer Features

Compare the cross-validated AUC of the churn pipeline with all seven features and with only `tenure_months` and `monthly_contract`.

In [ ]:
# Write your code below

### In-Class Exercise 3: Degree Curve

For degrees 1 to 12, compute the mean training and validation R2 with `cross_validate` on `bikes.sample(60, random_state=0)`. Plot both lines against the degree.

In [ ]:
# Write your code below

### In-Class Exercise 4: More Data Helps

Fit the degree 8 model on 25, 100 and 400 random training days (use `random_state=2`), always testing on the remaining days. How does the test R2 change?

In [ ]:
# Write your code below

## Summary

- One split gives a noisy score; cross-validation averages several.
- `cross_val_score(model, X, y, cv=5, scoring=...)` returns one score per fold.
- Overfitting: great on training data, poor on new data.
- Compare training and validation scores to spot the gap.
- Pipelines keep scaling inside each fold, preventing leakage.
- Simpler models, more data and regularisation reduce overfitting.

## What's Next

Next session is Project 3. You will frame a real decision, prepare a raw table, compare models with cross-validation, choose a threshold, score the test set once and explain the model. Bring your churn and apartment code from Sessions 20 to 22.